# NMDesc escape disease-gene paper — reproducible Colab runner

Runs the R analysis in [`CobanAkdemirlab/NMDescapediseasegene_paper`](https://github.com/CobanAkdemirlab/NMDescapediseasegene_paper) (v4 directories) on Google
Colab. Nothing to install locally; no Google account data required beyond signing in to
Colab.

**Status: not yet executed on Colab.** The dependency list and the path layer were
derived by scanning the 139 v4 scripts; install times and per-script runtimes are
unverified. Run order below is the intended one.

### What you need
Nothing, if `DATA_URL` below points at a published archive of the inputs. The repository
is code only — the v4 scripts read 352 distinct input files that are not tracked in git.
All of them derive from public sources (ClinVar, gnomAD, Ensembl, OMIM).

### Runtime
Python 3, CPU. R is called through `rpy2` cell magics. No GPU is used — this pipeline is
CPU-bound R.

## 1. Data

`DATA_URL` should be a single archive (`.tar.gz` or `.zip`) containing the input files,
flat or nested — the path layer resolves by basename. A Zenodo record or a GitHub release
asset both work and need no authentication.

If you are the author and have the inputs on your own machine, the companion script
`make_data_bundle.sh` builds this archive from `colab_input_inventory.csv`.

Leave `DATA_URL` empty to fall back to mounting your own Drive.

In [ ]:
DATA_URL = ""   # e.g. "https://zenodo.org/records/<id>/files/nmdesc_colab_data.tar.gz"
DATA_DIR = "/content/data"

import os, subprocess, sys
os.makedirs(DATA_DIR, exist_ok=True)

if DATA_URL:
    arch = "/content/" + DATA_URL.rstrip("/").split("/")[-1].split("?")[0]
    if not os.path.exists(arch):
        subprocess.run(["wget", "-q", "--show-progress", "-O", arch, DATA_URL], check=True)
    if arch.endswith((".tar.gz", ".tgz")):
        subprocess.run(["tar", "xzf", arch, "-C", DATA_DIR], check=True)
    elif arch.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", arch, "-d", DATA_DIR], check=True)
    else:
        sys.exit("DATA_URL must be .tar.gz or .zip")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/NMDesc_data"
    os.makedirs(DATA_DIR, exist_ok=True)

# Flatten, so nested archive layouts resolve by basename
n = 0
for dp, dn, fn in os.walk(DATA_DIR):
    if dp == DATA_DIR: continue
    for f in fn:
        dst = os.path.join(DATA_DIR, f)
        if not os.path.exists(dst):
            os.link(os.path.join(dp, f), dst); n += 1
print("data dir:", DATA_DIR)
print("files:", len([f for f in os.listdir(DATA_DIR)
                     if os.path.isfile(os.path.join(DATA_DIR, f))]), f"({n} linked up from subdirs)")

## 2. R packages

CRAN packages install from a **dated** Posit Package Manager snapshot, so the versions
resolved here are the same ones a reader gets next year. Change `SNAPSHOT` to move the
pin. Bioconductor is pinned by its own release, recorded in `sessionInfo()` at the end.

Two heavy dependencies are commented out; uncomment only if you run the stages that need
them. `BSgenome.Hsapiens.UCSC.hg38` is a ~700 MB download used by 4 scripts, and `brms`
pulls the Stan toolchain, also used by 4 scripts.

In [ ]:
SNAPSHOT = "2026-08-01"   # Posit PPM snapshot date; pins all CRAN versions
!apt-get -qq install -y r-base-core libcurl4-openssl-dev libxml2-dev libssl-dev libfontconfig1-dev > /dev/null
%load_ext rpy2.ipython
codename = subprocess.run(['bash','-lc','lsb_release -cs'],
                          capture_output=True, text=True).stdout.strip()
print("Ubuntu:", codename, "| CRAN snapshot:", SNAPSHOT)

In [ ]:
%%R -i codename -i SNAPSHOT
options(repos = c(CRAN = sprintf(
          "https://packagemanager.posit.co/cran/__linux__/%s/%s", codename, SNAPSHOT)),
        Ncpus = parallel::detectCores())
cran <- c("dplyr","ggplot2","tidyverse","readr","stringr","ggpubr","patchwork","tidyr",
          "data.table","scales","gt","gtsummary","readxl","lme4","lmerTest","rstatix",
          "flextable","here","DiscreteFDR","jsonlite","remotes","BiocManager")
need <- setdiff(cran, rownames(installed.packages()))
if (length(need)) install.packages(need)
still <- setdiff(cran, rownames(installed.packages()))
cat(if (length(still)) paste("CRAN FAILED:", paste(still, collapse=", ")) else
    "all CRAN packages present", "\n")

In [ ]:
%%R
bioc <- c("biomaRt","GenomicRanges","IRanges","S4Vectors","Biostrings","AnnotationDbi",
          "GenomicFeatures","VariantAnnotation")
need <- setdiff(bioc, rownames(installed.packages()))
if (length(need)) BiocManager::install(need, ask = FALSE, update = FALSE)
still <- setdiff(bioc, rownames(installed.packages()))
cat("Bioconductor", as.character(BiocManager::version()), "|",
    if (length(still)) paste("FAILED:", paste(still, collapse=", ")) else "all present", "\n")

# aenmd is not on CRAN or Bioconductor: 3 scripts call library(aenmd), 11 use aenmd::
if (!requireNamespace("aenmd", quietly = TRUE)) {
  remotes::install_github("kostkalab/aenmd.data.ensdb.v105", upgrade = "never")
  remotes::install_github("kostkalab/aenmd", upgrade = "never")
}
cat("aenmd:", requireNamespace("aenmd", quietly = TRUE), "\n")

# BiocManager::install("BSgenome.Hsapiens.UCSC.hg38", ask = FALSE, update = FALSE)
# install.packages("brms")

## 3. Repository and path layer

The v4 scripts carry 99 absolute path references across 13 prefixes on the original
authors' machines (`~/Desktop/...`, `/Users/jxu14/Desktop/...`,
`/Users/qkelly/Desktop/...`). `run_script()` substitutes them with `DATA_DIR` on a
working copy and sources that copy, leaving the checkout untouched; `setwd()` calls are
commented out on the copy. Prefix coverage was checked against the repository: all 99
references match a prefix.

In [ ]:
!git clone -q https://github.com/CobanAkdemirlab/NMDescapediseasegene_paper.git /content/NMDescapediseasegene_paper
os.environ["NMDESC_DATA"] = DATA_DIR
os.environ["NMDESC_REPO"] = "/content/NMDescapediseasegene_paper"
os.environ["NMDESC_WORK"] = "/content/work"
!cd /content/NMDescapediseasegene_paper && git rev-parse --short HEAD

In [ ]:
shim = r'''
# colab_paths.R -- path mapping layer for running this repo on Google Colab.
# The v4 scripts read data through absolute paths on the original author's
# machine. DATA_DIR points at one folder holding those inputs; run_script()
# rewrites the prefixes on a working copy and sources that copy.

DATA_DIR <- Sys.getenv("NMDESC_DATA", "/content/drive/MyDrive/NMDesc_data")
REPO_DIR <- Sys.getenv("NMDESC_REPO", "/content/NMDescapediseasegene_paper")
WORK_DIR <- Sys.getenv("NMDESC_WORK", "/content/work")
dir.create(WORK_DIR, showWarnings = FALSE, recursive = TRUE)

# Longest prefixes first, so nested ones are not shadowed.
PREFIXES <- c(
  "~/Desktop/NMDescapediseasegene_paper-main/new_NMDesc/data",
  "/Users/jxu14/Desktop/NMDescapediseasegene_paper-main",
  "~/Desktop/new_clinvar/raw_data",
  "~/Desktop/new_clinvar/snv_list/list4",
  "~/Desktop/new_clinvar",
  "/Users/jxu14/Desktop/enrich",
  "/Users/jxu14/Desktop/autism",
  "/Users/qkelly/Desktop/clinvar",
  "~/Desktop/clinvar",
  "~/Downloads",
  "~/Desktop",
  "/Users/jxu14/Desktop",
  "/Users/qkelly/Desktop"
)

map_path <- function(x) {
  for (p in PREFIXES) {
    if (startsWith(x, p)) return(file.path(DATA_DIR, sub("^/+", "", substring(x, nchar(p) + 1))))
  }
  x
}

# Every data reference a script makes, resolved against DATA_DIR.
script_inputs <- function(script) {
  txt <- readLines(file.path(REPO_DIR, script), warn = FALSE)
  m <- regmatches(txt, gregexpr('"[^"]*\\.(csv|txt|tsv|rds|fasta|fa|xlsx)"', txt, ignore.case = TRUE))
  refs <- unique(gsub('"', "", unlist(m)))
  if (!length(refs)) return(data.frame(ref = character(), resolved = character(), exists = logical()))
  res <- vapply(refs, function(r)
    if (startsWith(r, "/") || startsWith(r, "~")) map_path(r) else file.path(DATA_DIR, basename(r)),
    character(1))
  data.frame(ref = refs, resolved = unname(res), exists = file.exists(unname(res)),
             row.names = NULL, stringsAsFactors = FALSE)
}

# Report what a script needs before spending time on it.
check_script <- function(script) {
  d <- script_inputs(script)
  cat(sprintf("%s: %d data refs, %d present, %d missing\n",
              script, nrow(d), sum(d$exists), sum(!d$exists)))
  if (any(!d$exists)) print(d[!d$exists, c("ref", "resolved")], row.names = FALSE)
  invisible(d)
}

# Rewrite absolute prefixes on a copy, then source the copy.
run_script <- function(script, echo = TRUE) {
  src <- file.path(REPO_DIR, script)
  txt <- readLines(src, warn = FALSE)
  for (p in PREFIXES) txt <- gsub(p, DATA_DIR, txt, fixed = TRUE)
  txt <- gsub("setwd\\(", "# setwd(", txt)
  dst <- file.path(WORK_DIR, gsub("[/ ]", "_", script))
  writeLines(txt, dst)
  source(dst, echo = echo, max.deparse.length = 200)
  invisible(dst)
}

cat("colab_paths.R loaded\n  DATA_DIR:", DATA_DIR, "\n  REPO_DIR:", REPO_DIR, "\n")
'''
open('/content/colab_paths.R','w').write(shim)
print(len(shim.splitlines()), 'lines written')

In [ ]:
%%R
source("/content/colab_paths.R")

## 4. Readiness audit

Resolves every data reference each script makes and reports which inputs are present.
Run this before any stage — a missing input is the usual reason one fails.

In [ ]:
%%R
scripts <- list.files(REPO_DIR, pattern = "[.][Rr]$", recursive = TRUE)
scripts <- scripts[!grepl("^backup/", scripts)]
rdy <- do.call(rbind, lapply(scripts, function(s) {
  d <- script_inputs(s)
  data.frame(script = s, n_ref = nrow(d), n_missing = sum(!d$exists))
}))
rdy$ready <- rdy$n_ref > 0 & rdy$n_missing == 0
cat("v4 scripts:", nrow(rdy),
    "| runnable now:", sum(rdy$ready),
    "| blocked:", sum(rdy$n_ref > 0 & !rdy$ready),
    "| no data refs:", sum(rdy$n_ref == 0), "\n\n")
print(head(rdy[order(-rdy$ready, rdy$n_missing), ], 15), row.names = FALSE)
write.csv(rdy, "/content/work/readiness.csv", row.names = FALSE)

In [ ]:
%%R
# Which inputs are still missing, ranked by how many scripts want them
miss <- do.call(rbind, lapply(scripts, function(s) {
  d <- script_inputs(s); d <- d[!d$exists, , drop = FALSE]
  if (!nrow(d)) NULL else data.frame(need = basename(d$ref))
}))
if (is.null(miss)) cat("nothing missing\n") else print(head(sort(table(miss$need),
                                                                 decreasing = TRUE), 30))

## 5. Run a stage

`run_analysis.R` at the repository root is the documented entry point. Start with a
script whose inputs are all present according to the audit.

In [ ]:
%%R
check_script("run_analysis.R")

In [ ]:
%%R
# run_script("run_analysis.R")
# run_script("gene level_v4/gene_main_dbh.R")
# run_script("variant level_v4/variant_main_DBH.R")

## 6. Outputs and the environment record

Colab discards `/content` when the runtime recycles. `sessionInfo()` is written next to
the outputs so the exact package versions behind a result stay attached to it.

In [ ]:
%%R
out <- "/content/work/output"; dir.create(out, showWarnings = FALSE, recursive = TRUE)
made <- setdiff(list.files(WORK_DIR, pattern = "[.](csv|pdf|png|rds|txt)$",
                           recursive = TRUE, full.names = TRUE),
                list.files(out, recursive = TRUE, full.names = TRUE))
if (length(made)) file.copy(made, out, overwrite = TRUE)
writeLines(capture.output(sessionInfo()), file.path(out, "sessionInfo.txt"))
cat("outputs:", length(made), "->", out, "\n")

In [ ]:
from google.colab import files
!cd /content/work && tar czf /content/nmdesc_output.tar.gz output
files.download('/content/nmdesc_output.tar.gz')